# Acquisition des données

## Partie A : Import de fonctions utiles à l'importation des données

Avant de procéder à l'acquisition des données, il est essentiel de configurer notre environnement de travail. Dans une optique de **reproductibilité** et de **clarté du code**, la logique technique (les fonctions de téléchargement) a été déportée dans un dossier dédié nommé `/fonctions`, composé de fichiers Python. 

La cellule suivante initialise cette connexion en ajoutant le dossier des scripts au chemin système de Python et en important l'ensemble des dépendances nécessaires.

In [2]:
import sys
import os
import pandas as pd

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

# On importe toutes les fonctions dans le fichier imports.py
from imports import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[09/11/26 17:47:04] INFO     No custom team name replacements found. You can configure these in       _config.py:92
                             C:\Users\LouisHarle\soccerdata\config\teamname_replacements.json.                     

                    INFO     No custom league dict found. You can configure additional leagues in    _config.py:190
                             C:\Users\LouisHarle\soccerdata\config\league_dict.json.                               

## Partie B : Importation des données

Maintenant que nous avons importer les fonctions utiles à l'acquisition des données, il s'agit maintenant de les utiliser afin d'obtenir les bases de données qui seront utiles à notre modélisation afin de prédire les valeurs marchandes des joueurs masculins de football.

### B0 : Les paramètres à fixer

Il s'agit ici de définir les saisons et les ligues à analyser. Nous choisissons ici pour notre modélisation d'utiliser les saisons à partir de 2020/2021 jusque 2025/2026, et les ligues issues du Big 5, qui regroupent le plus de données, ce qui relèvera alrs d'une plus grande précision dans nos prédictions.

In [3]:
# On définit la période sur laquelle on souhaite récolter les données
annee_debut = 2020
annee_fin = 2025

# On définit les ligues que l'on souhaite analyser
ligues = {"GB1", "ES1", "L1", "IT1", "FR1"}

In [4]:
# TODO : pour le README mettre les nomenclatures des ligues

### B1 : Les données Transfermarkt

Ce dataset est une extraction des données du site de référence mondial **Transfermarkt**, mise à disposition via le projet open-source `player-scores` de David Cariboo. Il s'agit d'une base de données combinant des faits observés (transferts, compositions) et des **valeurs marchandes** validées par un réseau d'experts. Les valeurs sont mises à jour régulièrement.


Transfermarkt est en fait la référence de base pour l'évaluation financière des joueurs. Dans ce projet, cette source est alors **essentielle** car elle fournit notre **variable cible** : la valeur marchande.


Les fichiers utilisés ont donc chacun leur utilité propre afin de prédire la valeur marchande des joueurs de football. Ils contiennent l'historique temporel des prix, constituant ce que le modèle devra prédire. De plus, nous avons accès aux caractéristiques personnelles des joueurs : âge, taille, pied fort... 


A partir de recherches bibliographiques précédemment réalisées, nous pouvons émettre quelques hypothèses sur ces données Transfermarkt.
Par exemple, on s'attend à un pic des valeurs marchandes entre 24 et 28 ans avant un déclin lié à la valeur de revente. De plus, les joueurs issus de Premier League devraient être plus chers que les autres.

#### B1.1 : CONFIGURATION DE L'API KAGGLE

Avant d'utiliser cette fonction, vous devez créer un fichier `.env` à la racine de votre projet avec le format suivant :

KAGGLE_USERNAME=votre_nom_d_utilisateur

KAGGLE_API_TOKEN=votre_cle_api

**Où trouver ces informations ?**

*   **Identifiant :** Il s'agit de votre nom d'utilisateur Kaggle habituel.
*   **Token :** Allez sur votre profil Kaggle > **Settings** > section **API** > Cliquez sur **"Generate New API Token"**. Copiez ensuite le token dans le fichier `.env`.

In [5]:
# Configuration pour utiliser la fonction de téléchargement de données kaggle
DATASET = 'davidcariboo/player-scores'
DEST = "../data/transfermarkt_datasets"

# Appel de la fonction de téléchargement
download_kaggle_dataset(DATASET, DEST)

Téléchargement de davidcariboo/player-scores vers ../data/transfermarkt_datasets...
Dataset URL: https://www.kaggle.com/datasets/davidcariboo/player-scores
Téléchargement correctement effectué !


#### B1.2 : Conservation des données qui nous intéressent

Nous ne conservons que les fichiers concernant les informations intrinsèques aux différents joueurs, sur la période et les ligues choisies.

In [6]:
# Nous conservons les joueurs du Big 5 dont la dernière saison est au moins après la saison de départ choisie
players_path = os.path.join(DEST, "players.csv")
appearances_path = os.path.join(DEST, "appearances.csv")

players_filtered(players_path, appearances_path, annee_debut, annee_fin, ligues)

players.csv filtré : 7306 joueurs conservés.


In [7]:
# Nous conservons les valeurs marchandes réalisées après le 01/07 de la première saison sélectionnée
valuations_path = os.path.join(DEST, "player_valuations.csv")

valuations_filtered(valuations_path, appearances_path, annee_debut, annee_fin, ligues)

player_valuations.csv filtré : 45467 lignes conservées.


### B2 : Les données Soccerdata

Nous utilisons ici l'API de Soccerdata pour récolter des données issues de deux sources majeures du football européen : FBref et Understat. FBref fournit en effet des données de performances complètes et est la source de référence pour les statistiques avancées. Understat contient quant-à-lui des données plus poussées comme les **Expected Goals (xG)** ou encore les **Expected Assists (xA)**.


Nous utilisons ces 2 sources de données pour plusieurs raisons.
Tout d'abord, FBref offre un certain niveau de détail sur les actions défensives et la création de jeu (Progressive Carries, Tackles, etc.). De plus, Understat permet d'isoler la performance réelle de la "chance" ou de la "finition". Un joueur qui génère beaucoup de xG sans marquer reste une cible de transfert à fort potentiel. Enfin, l'API permet d'uniformiser les noms de ligues et les saisons entre les deux plateformes, réduisant le risque d'erreurs lors de la fusion.


Les variables ont été segmentées pour répondre aux besoins de notre futur modèle. Certaines évaluent le volume de jeu et l'influence tactique par poste tandis que d'autres mesurent la dangerosité et la contribution d'un joueur sur le terrain.


Nous pouvons supposer que les attaquants dont les `xG` sont élevés et constants affichent une valeur marchande supérieure, la capacité à se créer des occasions étant une compétence très valorisée sur le marché. Pour les créateurs, le `xA` est un meilleur prédicteur de la valeur que les passes décisives réelles, car il mesure la vision de jeu indépendamment du finisseur. Enfin, le championnat d'origine pourrait agir comme un coefficient multiplicateur sur la valeur marchande (prime liée à l'exposition financière de la Premier League par exemple).

In [8]:
# Nous récoltons ici les données d'Understat

df_xg = get_understat_data(annee_debut, annee_fin, ligues)
df_xg.to_csv(r'..\data\soccerdata\data_xg_soccerdata.csv', index=False, sep=',', encoding='utf-8-sig')

Récupération des données Understat pour les ligues : ['ITA-Serie A', 'ESP-La Liga', 'GER-Bundesliga', 'FRA-Ligue 1', 'ENG-Premier League'] et saisons : [2020, 2021, 2022, 2023, 2024, 2025]


[9/11/2026 5:47:50 PM] INFO     Saving cached data to C:\Users\LouisHarle\soccerdata\data\Understat  _common.py:250

[2026-09-11 17:47:52] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\.venv\Lib\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


[9/11/2026 5:47:52 PM] INFO     Successfully loaded TLS library: C:\Users\LouisHarle\OneDrive -    libraries.py:397
                                Quadratic\Bureau\Stage_VM\.venv\Lib\site-packages\tls_requests\bin                 
                                \tls-client-xgo-1.13.1-windows-amd64.dll                                           

                       WARNING  c:\Users\LouisHarle\OneDrive -                                  ]8;id=4387639;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py\_py_warnings.py]8;;\:]8;id=4387640;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py#230\230]8;;\
                                Quadratic\Bureau\Stage_VM\.venv\Lib\site-packages\soccerdata\_c                    
                                ommon.py:144: UserWarning: Season id "2021" is ambiguous:                          
                                interpreting as "20-21"                                                            
                                  warnings.warn(msg, stacklevel=1)                                                 
                                                                                                                   

16650 lignes récupérées


In [9]:
# Nous récoltons ici les données de FBref

data_advanced = get_fbref_data(annee_debut, annee_fin, ligues)
data_advanced.to_csv(r'..\data\soccerdata\data_soccerdata.csv', index=False, sep=',', encoding='utf-8-sig')

Récupération des données FBref pour les ligues : ['ITA-Serie A', 'ESP-La Liga', 'GER-Bundesliga', 'FRA-Ligue 1', 'ENG-Premier League'] et saisons : ['20-21', '21-22', '22-23', '23-24', '24-25', '25-26']


[9/11/2026 5:47:54 PM] INFO     Saving cached data to C:\Users\LouisHarle\soccerdata\data\FBref      _common.py:250



*** chromedriver to download = 153.0.8010.36 (Latest Stable) 

https://storage.googleapis.com/chrome-for-testing-public/153.0.8010.36/win64/chromedriver-win64.zip ...
Download Complete!

Extracting ['chromedriver.exe'] from chromedriver-win64.zip ...
Unzip Complete!

The file [uc_driver.exe] was saved to:
C:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\.venv\Lib\site-packages\seleniumbase\drivers\
uc_driver.exe

Making [uc_driver.exe 153.0.8010.36] executable ...
[uc_driver.exe 153.0.8010.36] is now ready for use!


*** chromedriver to download = 153.0.8010.36 (Latest Stable) 

https://storage.googleapis.com/chrome-for-testing-public/153.0.8010.36/win64/chromedriver-win64.zip ...
Download Complete!

Extracting ['chromedriver.exe'] from chromedriver-win64.zip ...
Unzip Complete!

The file [chromedriver.exe] was saved to:
C:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\.venv\Lib\site-packages\seleniumbase\drivers\
chromedriver.exe

Making [chromedriver.exe 153.0.8010.

[9/11/2026 5:48:15 PM] WARNING  c:\Users\LouisHarle\OneDrive -                                  ]8;id=4387645;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py\_py_warnings.py]8;;\:]8;id=4387646;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py#230\230]8;;\
                                Quadratic\Bureau\Stage_VM\.venv\Lib\site-packages\soccerdata\fb                    
                                ref.py:104: UserWarning: You are trying to scrape data for all                     
                                of the Big 5 European leagues. This can be done more                               
                                efficiently by setting leagues='Big 5 European Leagues                             
                                Combined'.                                                                         
                                  warnings.warn(                                                                   
                                                                                                                   

  Extraction : standard...
  Extraction : keeper...
  Extraction : shooting...
  Extraction : playing_time...
  Extraction : misc...
  Fusion 'keeper' : 40 colonnes
  Fusion 'shooting' : 51 colonnes
  Fusion 'playing_time' : 65 colonnes
  Fusion 'misc' : 75 colonnes
Format du dataset final : 17113 lignes et 75 colonnes


In [10]:
# Chargons le dataset FBref afin de le fusionner avec le dataset Understat
df_base = pd.read_csv(r'..\data\soccerdata\data_soccerdata.csv', sep=',', encoding='utf-8-sig')


df_final = merge_fbref_understat(
    df_base, 
    df_xg, 
    output_path=r'..\data\soccerdata\data_final_soccerdata.csv'
)

Fusion terminée : 17113 lignes et 80 colonnes.
Fichier sauvegardé sous : ..\data\soccerdata\data_final_soccerdata.csv


### B3 : Les données de blessures Transfermarkt

Les données de blessures peuvent fortement influencer la valeur marchande, notamment si un joueur a une longue absence suite à une blessure grave. Nous récoltons alors les données de blessures sur le sitre *Transfermarkt*.

Les données ont été récoltée via un script de **scraping** Python. Cette collecte parcourt les archives de chaque saison pour identifier les clubs présents en première division au moment des faits. Elle récolte alors les données de blessures des joueurs présents dans ces équipes.

Ces données sont très utiles car elles permettent d'obtenir la date, la nature précise de la blessure et le **nombre de matchs manqués**. La nature de la blessure est importante car un joueur ne jouant pas pendant une longue période pour une blessure grave verra sa valeur marchande fortement impactée. Nous obtenons alors l'ensemble des blessures des joueurs évoluant en première division des 5 grands championnats, de 2020 à 2025.


L'objectif est de déterminer dans quelle mesure l'historique médical impacte la valeur marchande d'un joueur. Nous supposons qu'une augmentation du ratio `Jours de blessure / Saison` entraîne une baisse significative de la valeur marchande par exemple. Les blessures dites "lourdes" (ligaments croisés, rupture du tendon d'Achille) devraient avoir un impact durable et plus sévère sur la valeur marchande que les blessures musculaires répétitives, même si ces dernières totalisent un nombre de jours d'absence identique.


Un joueur affichant une saison complète sans aucune blessure dans le "Big 5" devrait voir sa valeur marchande plus élevée par rapport à ses statistiques pures.

Les blessures n'agissent pas seulement directement sur la valeur marchande, mais aussi indirectement en diminuant le temps de jeu effectif, ce qui diminue ses statistiques de la saison (buts, passes décisives), alors que ceux-ci déterminent grandement la valeur marchande.

In [11]:
# Chargement des données de blessure des joueurs via le scraping du site Transfermarkt.

df_blessures = scraping_blessures_transfermarkt(
    annee_debut,
    annee_fin,
    "../data/dataset_blessures.csv",
    ligues=ligues,
    max_threads=12
)

Cartographie clubs et joueurs (2020 à 2025)

  Saison 2020/2021
    [IT1] 20 clubs trouvés
      Erreur réseau (tentative 1): HTTPSConnectionPool(host='www.transfermarkt.com', port=443): Read timed out. (read timeout=15)
      Erreur réseau (tentative 1): HTTPSConnectionPool(host='www.transfermarkt.com', port=443): Read timed out. (read timeout=15)
    [ES1] 20 clubs trouvés
      Erreur réseau (tentative 1): HTTPSConnectionPool(host='www.transfermarkt.com', port=443): Read timed out.
    [L1] 18 clubs trouvés
      Erreur réseau (tentative 1): HTTPSConnectionPool(host='www.transfermarkt.com', port=443): Read timed out. (read timeout=15)
      Erreur réseau (tentative 2): HTTPSConnectionPool(host='www.transfermarkt.com', port=443): Read timed out. (read timeout=15)
      Erreur réseau (tentative 1): HTTPSConnectionPool(host='www.transfermarkt.com', port=443): Read timed out. (read timeout=15)
      Erreur réseau (tentative 2): HTTPSConnectionPool(host='www.transfermarkt.com', port=443)

### B4 : Les données collectives du site *football-data.co.uk*

Ce site collecte les résultats détaillés des matchs de football (scores, statistiques de match) ainsi que les **cotes de paris sportifs** provenant des principaux bookmakers mondiaux, que nous n'utiliserons pas.


La valeur d'un joueur est fortement influencée par la **force de son équipe** et la difficulté de son championnat.Intégrer ces données permet de pondérer les performances individuelles par le niveau collectif de l'équipe du joueur.


Les données de *football-Data.co.uk* apportent une dimension contextuelle essentielle à l’analyse de la valeur marchande. Elles permettent de relier les performances individuelles à l’environnement collectif dans lequel évolue le joueur.

Les résultats des matchs et les statistiques collectives permettent d’évaluer la dynamique d’une équipe, son niveau de domination et sa capacité à créer des occasions. Un joueur évoluant dans une équipe performante et régulière bénéficie généralement d’une meilleure valorisation.

On peut alors supposer que la valeur marchande évolue en fonction de ces différentes caractéristiques collectives. Une corrélation positive est attendue entre le ratio de victoires et la hausse de la valeur marchande lors de la mise à jour suivante sur Transfermarkt. Le championnat pèse également dans la balance : des recherches bibliographiques nous ont mené à supposer l'hypothèse que les joueurs de Premier League sont plus chers que ceux des autres ligues du Big 5, même pour des équipes de bas de tableau.

In [12]:
# Importations des données collectives

download_football_data_datasets(annee_debut, annee_fin, ligues)

Impossible de télécharger I1 pour la saison 2021 : Remote end closed connection without response
Impossible de télécharger I1 pour la saison 2122 : Remote end closed connection without response
Impossible de télécharger I1 pour la saison 2223 : Remote end closed connection without response
Téléchargement réussi pour I1 saison 2324 : 380 lignes
Téléchargement réussi pour I1 saison 2425 : 380 lignes
Téléchargement réussi pour I1 saison 2526 : 380 lignes
Téléchargement réussi pour SP1 saison 2021 : 380 lignes
Téléchargement réussi pour SP1 saison 2122 : 380 lignes
Téléchargement réussi pour SP1 saison 2223 : 380 lignes
Téléchargement réussi pour SP1 saison 2324 : 380 lignes
Impossible de télécharger SP1 pour la saison 2425 : Remote end closed connection without response
Téléchargement réussi pour SP1 saison 2526 : 380 lignes
Téléchargement réussi pour D1 saison 2021 : 306 lignes
Téléchargement réussi pour D1 saison 2122 : 306 lignes
Téléchargement réussi pour D1 saison 2223 : 306 lignes
T

In [13]:
# Calcul des classements de fin de saison pour les ligues du Big 5

classements_fin_saison(input_filepath = "../data/football_data.csv",
                       output_filepath = "../data/classement_fin_saison.csv")

[9/11/2026 8:28:04 PM] WARNING  c:\Users\LouisHarle\OneDrive -                                  ]8;id=4387651;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py\_py_warnings.py]8;;\:]8;id=4387652;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py#230\230]8;;\
                                Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\imports                    
                                .py:819: DtypeWarning: Columns (0: Referee) have mixed types.                      
                                Specify dtype option on import or set low_memory=False.                            
                                  df = pd.read_csv(input_filepath)                                                 
                                                                                                                   

                       WARNING  c:\Users\LouisHarle\OneDrive -                                  ]8;id=4387657;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py\_py_warnings.py]8;;\:]8;id=4387658;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py#230\230]8;;\
                                Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\imports                    
                                .py:829: PerformanceWarning: DataFrame is highly fragmented.                       
                                This is usually the result of calling `frame.insert` many                          
                                times, which has poor performance.  Consider joining all                           
                                columns at once using pd.concat(axis=1) instead. To get a                          
                                de-fragmented frame, use `newframe = frame.copy()`                                 
                                  df["season"] = (                                                                 
                                                                                                                   

Le fichier '../data/classement_fin_saison.csv' a été généré avec succès !


,nom_equipe,ligue,saison,classement
51,Bayern Munich,D1,2020/2021,1
369,RB Leipzig,D1,2020/2021,2
130,Dortmund,D1,2020/2021,3
490,Wolfsburg,D1,2020/2021,4
137,Ein Frankfurt,D1,2020/2021,5
...,...,...,...,...
357,Osasuna,SP1,2025/2026,16
287,Mallorca,SP1,2025/2026,17
241,Levante,SP1,2025/2026,18
188,Girona,SP1,2025/2026,19


### B5 : Les données de classement FIFA

Nous avons téléchargé au préalable un dataset de l'historique du classement FIFA jusque le 1er avril 2026. Nous avons ensuite déterminé le top 10 à l'issue de chaque saison.

Le but est de constater si nos prédictions s'adaptent au contexte des compétitions internationales jouées par les joueurs ou non.

Le dataset vient du lien suivant : https://www.kaggle.com/datasets/cashncarry/fifaworldranking/discussion/687166

In [14]:
#Import du dataset

df_fifa = pd.read_csv(r'..\data\classement_fifa\fifa_ranking-2026-04-01.csv', sep=',', encoding='utf-8-sig')

In [15]:
# Nous extrayons le top 10 du classement FIFA 

df_fifa_top10 = extraire_classement_fin_saison(df_fifa, nombre_equipes=10, annee_debut=annee_debut, annee_fin=annee_fin)

# Annexes

Cette partie concerne des données récoltées initialement mais qui ne seront pas utilisées.

## FBref

FBref fournit des statistiques de performance agrégées par saison, principalement issues des données **Opta**. Contrairement aux données transactionnelles de Transfermarkt, FBref se concentre directement sur le jeu en tant que tel.

En effet, si Transfermarkt donne les valeurs marchandes, FBref donne les **justificatifs de performance** qui expliquent ce prix. Ces statistiques agissent comme des **variables explicatives** fondamentales. Elles permettent de distinguer un joueur efficace par chance d'un joueur créant régulièrement des occasions de haute qualité, ce qui influence directement sa valeur marchande.


Les données de FBref permettent d’analyser la valeur marchande des joueurs à travers leurs performances sportives.

Le temps de jeu renseigne sur la régularité, la disponibilité et l’importance d’un joueur dans son équipe, des éléments directement liés à sa valorisation. Les performances offensives permettent d’évaluer son efficacité et sa capacité à être décisif.

Les indicateurs collectifs mesurent son influence sur les résultats de l’équipe, tandis que les statistiques spécifiques aux gardiens permettent d’évaluer leur fiabilité défensive. Enfin, les données défensives et disciplinaires complètent l’analyse en mettant en évidence l’impact du joueur dans les duels et les phases défensives.


On peut s'attendre à des effets diverses sur la valeur marchandes. Par exemple, les joueur toujours titulaires devraient avoir une valeur marchande supérieure aux autres joueurs, tandis qu'une accumulation de cartons rouges ou de fautes commises sans volume défensif associé pourrait agir comme un malus sur la valeur marchande.

In [ ]:
# Configuration pour utiliser la fonction de téléchargement de données kaggle
DATASET = 'hubertsidorowicz/football-players-stats-2025-2026'
DEST = "../data/fbref_datasets"

# Appel de la fonction de téléchargement
download_kaggle_dataset(DATASET, DEST)

Téléchargement de hubertsidorowicz/football-players-stats-2025-2026 vers ../data/fbref_datasets...
Dataset URL: https://www.kaggle.com/datasets/hubertsidorowicz/football-players-stats-2025-2026
Téléchargement correctement effectué !


## Statsbomb

StatsBomb propose un accès "Open Data" à une partie de ses bases de données professionnelles. Contrairement aux sources précédentes, il s'agit de **données d'événements**. Chaque ligne représente une action technique précise (passe, tir, tacle, pression) géolocalisée sur le terrain via des coordonnées $(x, y)$.


Cette source est indispensable pour capturer le profil technique du joueur. Là où FBref nous dit qu'un joueur a réussi une passe, StatsBomb nous permet de calculer la difficulté de cette passe (distance, angle, nombre d'adversaires éliminés). Cela permet d'identifier des joueurs dont les statistiques classiques sont modestes mais dont la contribution technique à la progression du ballon est importante, justifiant ainsi des valeurs marchandes élevées ou en devenir.


Les données Statsbomb sont alors plus précises et permettent de percevoir des aspects techniques invisibles par d'autres données. En effet, ces données répertorient sur plusieurs compétitions et plusieurs saisons una analyse technique approfondie (pressions subies, longueurs de passes, localisations $x,y$). On identifie aussi les postes précis occupés par les joueurs sur le terrain en mesurant leur impact tactique, en sachant le contexte de chaque rencontre (adversaire, stade, date).


Nous pouvons supposer que les joueurs affichant un taux de réussite élevé sous pression possèdent une valeur marchande supérieure, car cette compétence est rare et recherchée par les clubs d'élite. De plus, on s'attend à ce que les joueurs capables de réaliser des passes progressives (brisant des lignes) dans le dernier tiers du terrain voient leur valeur augmenter plus rapidement que les joueurs effectuant des passes latérales sécurisées. Enfin, les coordonnées des actions permettront de valider si un joueur s'approche souvent de la surface adverse, augmentant mécaniquement son attractivité financière.

Téléchargons maintenant l'ensemble des données issues de l'Open source de Statsbomb. Les fichiers sont récoltés sous le format .json pour le moment.

In [ ]:
# Configuration pour utiliser la fonction de téléchargement de données git
REPO = "https://github.com/statsbomb/open-data"
DEST = "../data/statsbomb_datasets"

# Appel de la fonction de téléchargement
download_github_dataset(REPO, DEST) # type: ignore

Le dossier existe déjà. Vérification des mises à jour...
Les données sont déjà à jour !


Il s'agit maintenant de transformer ces données en fichiers .feather, plus pratique pour nous analyses futures.

In [ ]:
# Dossier où on stocke les fichiers finaux
destination = "../data/statsbomb_datasets/data"

In [ ]:
# Les compétitions

compile_statsbomb_to_feather(
    json_folder_path="../data/statsbomb_datasets/data", 
    output_folder_path=destination,
    output_name="competitions_statsbomb",
    recursive=False
)

Traitement de 1 fichiers trouvés dans data...


Fusion et sauvegarde...
Terminé ! Fichier : competitions_statsbomb.feather (80 lignes)


In [ ]:
# Les lineups

compile_statsbomb_to_feather(
    json_folder_path="../data/statsbomb_datasets/data/lineups", 
    output_folder_path=destination,
    output_name="all_lineups",
    record_path=['lineup'],
    meta=['team_name', 'team_id'],
    recursive = False
)

Traitement de 4235 fichiers trouvés dans lineups...
Fusion et sauvegarde...
Terminé ! Fichier : all_lineups.feather (161958 lignes)


In [ ]:
# Les events

# On ne garde que les colonnes essentielles car il y a trop d'informations dans ces fichiers.
cols_events = [
    'match_id', 'id', 'index', 'period', 'timestamp', 'minute', 'second', 
    'type.name', 'team.name', 'player.name', 'position.name', 
    'location', 'duration', 'under_pressure', 'pass.end_location', 
    'pass.outcome.name', 'shot.statsbomb_xg', 'shot.outcome.name'
]

compile_statsbomb_to_feather(
    json_folder_path="../data/statsbomb_datasets/data/events", 
    output_folder_path=destination,
    output_name="all_events",
    columns_to_keep=cols_events,
    recursive = False
)

Traitement de 4235 fichiers trouvés dans events...
Fusion et sauvegarde...
Terminé ! Fichier : all_events.feather (14874171 lignes)


In [ ]:
# Les matches

# Les colonnes essentielles pour les matches
cols_matches = [
    'match_id', 'match_date', 'kick_off', 'competition.competition_name', 
    'season.season_name', 'home_team.home_team_name', 'away_team.away_team_name', 
    'home_score', 'away_score'
]


compile_statsbomb_to_feather(
    json_folder_path="../data/statsbomb_datasets/data/matches", 
    output_folder_path=destination,
    output_name="all_matches",
    columns_to_keep=cols_matches,
    recursive = True
)

Traitement de 80 fichiers trouvés dans matches...
Fusion et sauvegarde...
Terminé ! Fichier : all_matches.feather (3961 lignes)


Les fichiers JSON volumineux ont été transformés en format .feather pour optimiser la vitesse de lecture et l'usage de la mémoire RAM.